# KG catch-up + intent chat session -> turns + knowledge graph

A new process flow on top of **[`kg_intent_chat.ipynb`](kg_intent_chat.ipynb)**: instead of
starting cold, the agent first figures out **how long it's been since it last spoke with this
human, what they'd talked about back then, and how MUCH of that it should expect to hear about
again** for a gap this long -- then keeps the conversation going, topic by topic, until it judges
it has *enough* for the gap period, not just "asked once about everything."

Built on **[`gaps_from_kg/get_temporal_containers.py`](../src/cltl/gaps_from_kg/get_temporal_containers.py)**
-- a different query layer over the same GraphDB repository (via `cltl.brain.LongTermMemory`)
than `kg_gap_finder.py`/`intent_gap_finder.py`'s rdflib/SPARQL-endpoint queries, which is what the
rest of the per-turn flow below still uses unchanged. See
**[`catch_up_from_kg.py`](catch_up_from_kg.py)** for the full implementation; in short, a loop:

1. **Ask about a gap-period topic** (`SaturationTracker.next_question()`) -- one of
   `chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES` (exercise, diet, sleep, symptoms, medication, ...)
   this human has real history with, but not yet ENOUGH reported for this gap (see "Saturation"
   below).
2. **Whatever they report is handled entirely by `KgIntentChatSession`'s own EXISTING per-turn
   flow, unchanged**: SRL extraction -> push to the KG -> `intent_gap_finder.next_intent_gap()`
   keeps asking its own follow-up questions about *that* activity (what/how much/when/where) for
   as long as its matching intent still has unmet requirements.
3. **Once that activity's own follow-ups are exhausted**, go back to step 1 -- another
   under-covered topic, or the same one again if it's still short -- **unless every topic has
   reached saturation**, at which point the conversation just continues normally.

**Saturation** -- "enough knowledge for the gap period" -- is defined *per topic* as the average
frequency of that topic in periods the same length as the gap, found by tiling that window size
backwards across this human's own history (`_windowed_average_rate()`): if they've historically
reported exercise ~3 times per week-long period and the gap is a week, 3 reported live is enough
for exercise specifically -- not "ask about it once and move on" and not "keep asking forever."
A hard per-topic cap (`SaturationTracker.MAX_ASKS_PER_TOPIC`, default 3) still gives up on a topic
the human simply has nothing more to say about, regardless of its target.

**Before running this:** same requirements as `kg_intent_chat.ipynb` -- `OPENAI_API_KEY` set, and
a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox` repository) at
`KG_ADDRESS` below, ideally one that already has some history for `HUMAN` in it (otherwise there's
nothing to catch up on, and this degrades to a plain, short "what's new?" opener with an
immediately-saturated tracker).

In [1]:
import time
from datetime import datetime, timedelta

from chat_sessions import KgIntentChatSession, openai_agent, save_turns
import catch_up_from_kg as catch_up

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"

# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()).
INTENTS_DIR = None

HUMAN = "Mehmet"

# A FRESH id every run, not a fixed constant -- see kg_intent_chat.ipynb's own CHAT_ID cell for
# why (reusing one across runs collides every run's activities on the same subject URIs).
CHAT_ID = int(time.time())

# "Now", for both the catch-up queries below and every activity pushed to the graph this run.
CURRENT_DATE = datetime.now()

# Used only if the KG has no earlier conversation with HUMAN on record at all (e.g. a brand new
# human, or a fresh/empty graph) -- how far back to assume the last (nonexistent) conversation
# was, so there's still a sensible "it's been N days" opener instead of a crash.
FALLBACK_LAST_CONVERSATION_DATE = CURRENT_DATE - timedelta(days=7)


## Step 1: how long has it been, and what's worth catching up on?

Connects to the same KG `KG_ADDRESS` points at (via `cltl.brain.LongTermMemory` this time, not
rdflib). `connect_brain()` also makes sure the graph has
**[`gaps_from_kg/n2mu_sem_roles.py`](../src/cltl/gaps_from_kg/n2mu_sem_roles.py)**'s small
`rdfs:subPropertyOf` mapping uploaded (a one-time, idempotent step -- see that module's own
docstring): without it, every real activity looks dateless/actorless/placeless to
`get_temporal_containers()`'s underlying query, since activities only ever carry this project's
own `n2mu:agent`/`n2mu:location`/`n2mu:time/...` predicates, never `sem:hasActor`/`hasPlace`/
`hasTime` directly. Past that one-time schema write, the rest of this step only reads.

In [2]:
brain = catch_up.connect_brain(KG_ADDRESS, log_dir=KG_LOG_DIR)

last_conversation_date = catch_up.find_last_conversation_date(
    HUMAN, brain, CURRENT_DATE, FALLBACK_LAST_CONVERSATION_DATE
)
catch_up_topics = catch_up.find_catch_up_topics(brain, CURRENT_DATE, last_conversation_date)

print(f"Last conversation with {HUMAN}: {last_conversation_date} "
      f"({(CURRENT_DATE.date() - last_conversation_date.date()).days} day(s) ago)")
print(f"Found {len(catch_up_topics)} catch-up topic(s):")
for topic in catch_up_topics:
    print(f"  - {topic['activity_type']}: target {topic['expected_count']} for this gap "
          f"(already {topic['initial_reported_count']} logged), {topic['history_count']} past "
          f"mention(s), most recently \"{topic['latest_label']}\" on {topic['latest_date']}")


2026-09-17 21:23:44 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-09-17 21:23:44 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-09-17 21:23:44 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-09-17 21:23:44 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-09-17 21:23:44 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Our previous conservation was on Thursday 2026-09-10 21:23:35.677984
Today is Thursday 2026-09-17 21:23:35.677984
What happened in the last 7 days?
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:exercise.                 ?id rdfs:label ?label .                 }
I found 29 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_food.                 ?id rdfs:label ?label .                 }
I found 12 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_drink.                 ?id rdfs:label ?label .                 }
I found 13 activities
PREFIX n2mu: <http://cltl.nl/l

## Step 2: open the chat

`SaturationTracker.opening_question()` turns the findings above into the agent's very first turn
(recorded via `open_with()`, same as `kg_chat_session.ipynb`'s own opening-greeting cell -- see
there for why it's a real, annotated-and-pushed turn rather than just printed text), naming the
`LEAD_TOPICS` topics with the biggest shortfall as concrete memory prompts, and marks those as
asked once so the loop below doesn't immediately repeat them.

`wrap_agent_fn_with_saturation_loop()` then wraps the plain `openai_agent()` default reply
function: for as long as `tracker.is_saturated()` is False, a reply that would otherwise be the
generic default LLM one asks about the next under-covered topic instead -- see
`catch_up_from_kg.py`'s own module docstring for exactly when that fires versus an ordinary
intent-driven gap question, and how `on_new_subject=tracker.record_new_activity` keeps the
tracker's own counts up to date as the human reports things live.

In [3]:
from kg_chat_gui import run_gui

LEAD_TOPICS = 2
tracker = catch_up.SaturationTracker(catch_up_topics, human=HUMAN)
agent_fn = catch_up.wrap_agent_fn_with_saturation_loop(openai_agent(), tracker)

kg_session = KgIntentChatSession(
    chat=CHAT_ID,
    human=HUMAN,
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    intents_dir=INTENTS_DIR,
    agent_fn=agent_fn,
    on_new_subject=tracker.record_new_activity,
)
print(f"Loaded {len(kg_session.intents)} intent(s) covering activity types: "
      f"{sorted({t for intent in kg_session.intents for t in intent.get('activity_types', [])})}")

opening_question = tracker.opening_question(CURRENT_DATE, last_conversation_date, lead_topics=LEAD_TOPICS)
print("saturation targets:", {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})

kg_session.open_with(opening_question)
kg_turns = run_gui(kg_session)


2026-09-17 21:24:01 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Loaded 15 intent(s) covering activity types: ['economic_condition', 'exercise', 'measurement', 'mental_condition', 'physical_condition', 'sleep', 'social_condition', 'symptom', 'take_drink', 'take_food', 'take_medicine']
1 catch-up topic(s) still queued after the opening turn: ['mental_condition']
turn {'chat': 1789673015, 'human': 'Mehmet', 'date': '2026,Sep,17', 'turn': 1, 'speaker': 'agent', 'utterance': 'Hi Mehmet, it’s nice to talk with you again—it’s been about a week since we last checked in. How have things been going with your eating lately, and how have your blood sugar measurements been over the past few days?'}


2026-09-17 21:24:04 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-17 21:24:04 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-09-17 21:24:04 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-09-17 21:24:04 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-09-17 21:24:04 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-09-17 21:24:04 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Compliance issues for chat 1789673015 turn 1:
 - extraction[0]: activity offset auto-corrected for 'eating': (137,6) -> (130,6)
 - extraction[0]: agent offset auto-corrected for 'we': (85,2) -> (62,2)
 - extraction[0]: qualification offset auto-corrected for 'lately': (144,6) -> (137,6)
 - extraction[1]: agent offset auto-corrected for 'we': (85,2) -> (62,2)
 - extraction[1]: time offset auto-corrected for 'over the past few days': (192,23) -> (193,22)
Total nr of scenarios 1


  0%|          | 0/1 [00:00<?, ?it/s]2026-09-17 21:24:05 -     INFO -                                    cltl.brain.LongTermMemory - Context: context1789673015


...

Compliance issues for chat 1789673015 turn 11:
 - extraction[0]: activity offset auto-corrected for 'letting me know': (23,14) -> (23,15)
 - extraction[0]: agent offset auto-corrected for 'me': (13,2) -> (31,2)
Total nr of scenarios 1


  0%|          | 0/1 [00:00<?, ?it/s]2026-09-17 21:26:57 -     INFO -                                    cltl.brain.LongTermMemory - Context: context1789673015
2026-09-17 21:26:57 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: letting me know_agent_agent [activity or social_->_person])
2026-09-17 21:26:57 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: letting me know_patient_agent [activity or social_->_person])


Conversation id 1789673015 Total number of capsules extracted for this conversation 1
chat 1789673015 out of  1 turn 11 out of 1 turns


100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


[turn 11] agent: No worries, thanks for letting me know.
    pushed 2 triple(s):
      letting me know  agent  =  me
      letting me know  patient  =  me
[kg_chat_gui] conversation saved to /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789673015_turns_20260917-212719.json
[kg_chat_gui] statistics saved to  /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789673015_stats_20260917-212719.json
[kg_chat_gui] gap log saved to     /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789673015_gaplog_20260917-212719.json


Inspect what was extracted, pushed, and where each agent reply came from:

In [4]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
print("saturated:", tracker.is_saturated())
print("reported vs. target, per topic:",
      {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})
print("catch-up questions actually asked:", tracker.asked_log)
kg_session.kg_pushes


11 turns, 10 annotated, 10 pushes to the knowledge graph
reply sources: ['gap', 'gap', 'gap', 'gap', 'gap']
catch-up topics actually asked about: []


[{'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1}]

`kg_session.turn_log` has the same per-turn breakdown `kg_intent_chat.ipynb` prints live: what
each turn pushed, which intent (if any) matched, and which requirement its reply was about
(`None` for a default- or saturation-loop-driven reply -- both are indistinguishable from
`turn_log`'s own point of view, since neither comes from `intent_gap_finder`;
`kg_session.reply_sources`/`tracker.asked_log` above are what tell them apart).

In [5]:
kg_session.turn_log

[{'turn': 1,
  'speaker': 'agent',
  'utterance': 'Hi Mehmet, it’s nice to talk with you again—it’s been about a week since we last checked in. How have things been going with your eating lately, and how have your blood sugar measurements been over the past few days?',
  'triples_pushed': [{'subject': 'eating',
    'predicate': 'agent',
    'object': 'we'},
   {'subject': 'eating', 'predicate': 'participant', 'object': 'Mehmet'},
   {'subject': 'eating', 'predicate': 'qualification', 'object': 'lately'},
   {'subject': 'blood sugar measurements',
    'predicate': 'agent',
    'object': 'we'},
   {'subject': 'blood sugar measurements',
    'predicate': 'participant',
    'object': 'Mehmet'},
   {'subject': 'blood sugar measurements',
    'predicate': 'time',
    'object': 'over the past few days'}],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 2,
  'speaker': 'Mehmet',
  'utterance': 'I went out with a friend and had a few drinks',
  'triples_pushed': [{'subject': 'went out w

In [6]:
save_turns(kg_session.turns, "kg_catchup_intent_turns.json")

Wrote 11 turns to kg_catchup_intent_turns.json
